In [1]:
%%capture
!pip install unsloth

In [2]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048
dtype = None  # auto-detect (Colab T4 = float16)
load_in_4bit = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/DeepSeek-R1-Distill-Qwen-1.5B",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.9.2: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

Unsloth: Will load unsloth/deepseek-r1-distill-qwen-1.5b-unsloth-bnb-4bit as a legacy tokenizer.


In [3]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                       "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

Unsloth 2026.9.2 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


In [4]:
from datasets import load_dataset

dataset = load_dataset("Magpie-Align/Magpie-Reasoning-V2-250K-CoT-Deepseek-R1-Llama-70B", split="train[:1000]")
print(dataset[0])

README.md:   0%|          | 0.00/7.65k [00:00<?, ?B/s]

data/train-00000-of-00009.parquet: reconstructing file:   0%|          |  0.00B /  167MB            

data/train-00000-of-00009.parquet: downloading bytes:           |  0.00B            

data/train-00001-of-00009.parquet: reconstructing file:   0%|          |  0.00B /  173MB            

data/train-00001-of-00009.parquet: downloading bytes:           |  0.00B            

...

data/train-00007-of-00009.parquet: downloading bytes:           |  0.00B            

data/train-00008-of-00009.parquet: reconstructing file:   0%|          |  0.00B /  190MB            

data/train-00008-of-00009.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/249922 [00:00<?, ? examples/s]

{'conversation_id': 'Llama-3.1-70B-Instruct_1', 'instruction': 'The duration of a process used to manufacture components is known to be normally distributed with a mean of 30 minutes and a standard deviation of 4 minutes. What is the probability of a time greater than 33 minutes being recorded?', 'response': "<think>\nOkay, so I have this problem here about the duration of a manufacturing process. It says the time is normally distributed with a mean of 30 minutes and a standard deviation of 4 minutes. I need to find the probability that the time is greater than 33 minutes. Hmm, let me think about how to approach this.\n\nFirst, I remember that in a normal distribution, the probability of a value being greater than the mean is 0.5 because the distribution is symmetric. But here, we're dealing with a specific value, 33 minutes, which is higher than the mean of 30 minutes. So, I need to figure out how far 33 is from the mean in terms of standard deviations and then use that to find the pr

In [5]:
dataset[0]

{'conversation_id': 'Llama-3.1-70B-Instruct_1',
 'instruction': 'The duration of a process used to manufacture components is known to be normally distributed with a mean of 30 minutes and a standard deviation of 4 minutes. What is the probability of a time greater than 33 minutes being recorded?',
 'response': "<think>\nOkay, so I have this problem here about the duration of a manufacturing process. It says the time is normally distributed with a mean of 30 minutes and a standard deviation of 4 minutes. I need to find the probability that the time is greater than 33 minutes. Hmm, let me think about how to approach this.\n\nFirst, I remember that in a normal distribution, the probability of a value being greater than the mean is 0.5 because the distribution is symmetric. But here, we're dealing with a specific value, 33 minutes, which is higher than the mean of 30 minutes. So, I need to figure out how far 33 is from the mean in terms of standard deviations and then use that to find the 

In [6]:
prompt_template = """Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
{}

### Response:
{}"""

EOS_TOKEN = tokenizer.eos_token

def formatting_prompts_func(examples):
    instructions = examples["instruction"]
    responses = examples["response"]
    texts = []
    for instruction, response in zip(instructions, responses):
        text = prompt_template.format(instruction, response) + EOS_TOKEN
        texts.append(text)
    return {"text": texts}

dataset = dataset.map(formatting_prompts_func, batched=True)
print(dataset[0]["text"][:500])

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
The duration of a process used to manufacture components is known to be normally distributed with a mean of 30 minutes and a standard deviation of 4 minutes. What is the probability of a time greater than 33 minutes being recorded?

### Response:
<think>
Okay, so I have this problem here about the duration of a manufacturing process. It says the time is normally distributed


In [7]:
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 60,
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
    ),
)

Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/1000 [00:00<?, ? examples/s]

In [8]:
trainer_stats = trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 1,000 | Num Epochs = 1 | Total steps = 60
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 18,464,768 of 1,795,552,768 (1.03% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss
1,0.530026
2,0.438596
3,0.498152
4,0.505469
5,0.545230
6,0.564812
7,0.389913
8,0.507846
9,0.388713
10,0.412763


Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-60/tokenizer_config.json.


In [10]:
from transformers import TextStreamer

text_streamer = TextStreamer(tokenizer, skip_prompt=True)

inputs = tokenizer(
    [prompt_template.format(
        "What is the derivative of x^3 + 2x?",
        ""
    )],
    return_tensors = "pt"
).to("cuda")

_ = model.generate(
    **inputs,
    streamer = text_streamer,
    max_new_tokens = 2048,
    use_cache = True,
)

Both `max_new_tokens` (=2048) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


<think>
Okay, so I have this problem: find the derivative of x cubed plus 2x. Hmm, I remember from class that when you take a derivative, you basically find the rate at which something is changing. So, if I have a function like x^3 + 2x, I need to find its derivative. I think I need to use the power rule here. Let me recall what the power rule is.

The power rule says that if you have a function f(x) = x^n, where n is any real number, then the derivative f'(x) is n times x raised to the power of (n - 1). So, for example, if I have f(x) = x^2, the derivative would be 2x. That makes sense because the slope of x squared increases as x increases.

But in this problem, I have two terms: x^3 and 2x. I think I need to find the derivative of each term separately and then add them together. That should give me the derivative of the entire function.

Let me start with the first term, x^3. Applying the power rule, the derivative should be 3x^(3 - 1), which is 3x squared. Okay, that seems straight

In [ ]:
model.push_to_hub_merged(
    "username/deepseek-r1-distill-qwen-1.5b-reasoning-finetune",
    tokenizer,
    save_method = "merged_16bit",
    token = "hf_MtGk...",
)

config.json:   0%|          | 0.00/679 [00:00<?, ?B/s]

Unsloth: Restored added_tokens_decoder metadata in Farhankhan1/deepseek-r1-distill-qwen-1.5b-reasoning-finetune/tokenizer_config.json.


Found HuggingFace hub cache directory: /root/.cache/huggingface/hub
Checking cache directory for required files...
Cache check failed: model.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Preparing safetensor model files:   0%|          | 0/1 [00:00<?, ?it/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.55GB            

...

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...inetune/model.safetensors:   0%|          | 15.4MB / 3.55GB            

Unsloth: Merging weights into 16bit: 100%|██████████| 1/1 [02:30<00:00, 150.97s/it]


Unsloth: Merge process complete. Saved to `/content/Farhankhan1/deepseek-r1-distill-qwen-1.5b-reasoning-finetune`


In [12]:
prompt_template.format(
        "What is the derivative of x^3 + 2x?",
        "")

'Below is an instruction that describes a task. Write a response that appropriately completes the request.\n\n### Instruction:\nWhat is the derivative of x^3 + 2x?\n\n### Response:\n'